# Bonus: Unit Commitment

PyPSA supports unit commitment constraints for its components, such as minimum part-load, minimum up time, minimum down time, start up costs, shut down costs and ramp rate restrictions. These constraints are important for accurately modeling the operation of conventional power plants.

This tutorial runs through examples of unit commitment for generators at a single bus.

To enable unit commitment on a component, set its attribute `committable` to `True`.

In [10]:
import pandas as pd

import pypsa

## Minimum Part Load

**Case description:** In the final snapshot, the load goes below the part-load limit of the coal generator (30%), forcing gas to commit.

In [11]:
nu = pypsa.Network(snapshots=range(4))

nu.add("Bus", "bus")

nu.add(
    "Generator",
    "coal",
    bus="bus",
    committable=True,
    p_min_pu=0.3,
    marginal_cost=20,
    p_nom=10_000,
)

nu.add(
    "Generator",
    "gas",
    bus="bus",
    committable=True,
    marginal_cost=70,
    p_min_pu=0.1,
    p_nom=1_000,
)

nu.add("Load", "load", bus="bus", p_set=[4_000, 6_000, 5_000, 800])

nu.optimize(log_to_console=False)

/tmp/ipykernel_866640/2867606449.py:27: FutureWarning:

The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.

Index(['bus'], dtype='object', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.model:Solver options:
 - log_to_console: False
INFO:linopy.io: Writing time: 0.04s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 32 primals, 60 duals
Objective: 3.56e+05
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-status-p-fixed-upper, Generator-start_up-p-fixed-upper, Generator-shut_down-p-fixed-upper, Generator-com-p-lower, Generator-com-p-upper, Generator-com-transition-start-up, Generator-com-transition-shut-down, Gene

('ok', 'optimal')

In [12]:
nu.generators_t.status

name,coal,gas
snapshot,,
0,1.0,0.0
1,1.0,0.0
2,1.0,0.0
3,0.0,1.0


In [13]:
nu.generators_t.p

name,coal,gas
snapshot,,
0,4000.0,0.0
1,6000.0,0.0
2,5000.0,0.0
3,0.0,800.0


## Minimum Up Time

**Case description:** Gas has a minimum up time, forcing it to be online longer than otherwise necessary, which incurs a standby cost for status up without generation.

In [14]:
nu = pypsa.Network(snapshots=range(4))

nu.add("Bus", "bus")

nu.add(
    "Generator",
    "coal",
    bus="bus",
    committable=True,
    p_min_pu=0.3,
    marginal_cost=20,
    p_nom=10000,
)

nu.add(
    "Generator",
    "gas",
    bus="bus",
    committable=True,
    stand_by_cost=50,
    marginal_cost=70,
    p_min_pu=0.1,
    up_time_before=0,
    min_up_time=3,
    p_nom=1_000,
)

nu.add("Load", "load", bus="bus", p_set=[4_000, 800, 5_000, 3_000])

nu.optimize(log_to_console=False)

/tmp/ipykernel_866640/1583081587.py:30: FutureWarning:

The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.

Index(['bus'], dtype='object', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.model:Solver options:
 - log_to_console: False
INFO:linopy.io: Writing time: 0.04s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 32 primals, 63 duals
Objective: 3.06e+05
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-status-p-fixed-upper, Generator-start_up-p-fixed-upper, Generator-shut_down-p-fixed-upper, Generator-com-p-lower, Generator-com-p-upper, Generator-com-transition-start-up, Generator-com-transition-shut-down, Gene

('ok', 'optimal')

In [15]:
nu.generators_t.status

name,coal,gas
snapshot,,
0,1.0,1.0
1,0.0,1.0
2,1.0,1.0
3,1.0,0.0


In [16]:
nu.objective

306150.0

In [17]:
nu.generators_t.p

name,coal,gas
snapshot,,
0,3900.0,100.0
1,0.0,800.0
2,4900.0,100.0
3,3000.0,-0.0


## Minimum Down Time

**Case description:** Coal has a minimum down time, forcing it to go off longer than otherwise cost-optimal.

In [18]:
nu = pypsa.Network(snapshots=range(4))

nu.add("Bus", "bus")

nu.add(
    "Generator",
    "coal",
    bus="bus",
    committable=True,
    p_min_pu=0.3,
    marginal_cost=20,
    min_down_time=2,
    down_time_before=1,
    p_nom=10_000,
)

nu.add(
    "Generator",
    "gas",
    bus="bus",
    committable=True,
    marginal_cost=70,
    p_min_pu=0.1,
    p_nom=4_000,
)

nu.add("Load", "load", bus="bus", p_set=[3_000, 800, 3_000, 8_000])

nu.optimize(log_to_console=False)

/tmp/ipykernel_866640/2744240505.py:29: FutureWarning:

The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.

Index(['bus'], dtype='object', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.model:Solver options:
 - log_to_console: False
INFO:linopy.io: Writing time: 0.04s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 32 primals, 64 duals
Objective: 4.86e+05
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-status-p-fixed-upper, Generator-start_up-p-fixed-upper, Generator-shut_down-p-fixed-upper, Generator-com-p-lower, Generator-com-p-upper, Generator-com-transition-start-up, Generator-com-transition-shut-down, Gene

('ok', 'optimal')

In [19]:
nu.objective

486000.0

In [20]:
nu.generators_t.status

name,coal,gas
snapshot,,
0,0.0,1.0
1,0.0,1.0
2,1.0,0.0
3,1.0,0.0


In [21]:
nu.generators_t.p

name,coal,gas
snapshot,,
0,0.0,3000.0
1,0.0,800.0
2,3000.0,0.0
3,8000.0,0.0


## Start Up and Shut Down Costs

**Case description:** Now there are costs associated with shut down and start up events, which could incentivise longer up times of generators with high start-up and shut-down costs.

In [22]:
nu = pypsa.Network(snapshots=range(4))

nu.add("Bus", "bus")

nu.add(
    "Generator",
    "coal",
    bus="bus",
    committable=True,
    p_min_pu=0.3,
    marginal_cost=20,
    min_down_time=2,
    start_up_cost=5_000,
    p_nom=10_000,
)

nu.add(
    "Generator",
    "gas",
    bus="bus",
    committable=True,
    marginal_cost=70,
    p_min_pu=0.1,
    shut_down_cost=25,
    p_nom=4_000,
)

nu.add("Load", "load", bus="bus", p_set=[3_000, 800, 3_000, 8_000])

nu.optimize(log_to_console=False)

/tmp/ipykernel_866640/3295327048.py:30: FutureWarning:

The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.

Index(['bus'], dtype='object', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.model:Solver options:
 - log_to_console: False
INFO:linopy.io: Writing time: 0.04s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 32 primals, 63 duals
Objective: 4.91e+05
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-status-p-fixed-upper, Generator-start_up-p-fixed-upper, Generator-shut_down-p-fixed-upper, Generator-com-p-lower, Generator-com-p-upper, Generator-com-transition-start-up, Generator-com-transition-shut-down, Gene

('ok', 'optimal')

In [23]:
nu.objective

491025.0

In [24]:
nu.generators_t.status

name,coal,gas
snapshot,,
0,0.0,1.0
1,0.0,1.0
2,1.0,0.0
3,1.0,0.0


In [25]:
nu.generators_t.p

name,coal,gas
snapshot,,
0,0.0,3000.0
1,0.0,800.0
2,3000.0,0.0
3,8000.0,0.0


## Ramp Rate Limits

Ramp rate limits can be set for ramping up and down and are given as percentage of the nominal power that can be ramped up or down per snapshot.

In [26]:
nu = pypsa.Network(snapshots=range(6))

nu.add("Bus", "bus")

nu.add(
    "Generator",
    "coal",
    bus="bus",
    marginal_cost=20,
    ramp_limit_up=0.1,
    ramp_limit_down=0.2,
    p_nom=10_000,
)

nu.add("Generator", "gas", bus="bus", marginal_cost=70, p_nom=4_000)

nu.add("Load", "load", bus="bus", p_set=[4_000, 7_000, 7_000, 7_000, 7_000, 3_000])

nu.optimize(log_to_console=False)

/tmp/ipykernel_866640/1927011143.py:19: FutureWarning:

The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.

Index(['bus'], dtype='object', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.model:Solver options:
 - log_to_console: False
INFO:linopy.io: Writing time: 0.02s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 12 primals, 40 duals
Objective: 9.50e+05
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Generator-p-ramp_limit_up, Generator-p-ramp_limit_down were not assigned to the network.


('ok', 'optimal')

In [27]:
nu.generators_t.p

name,coal,gas
snapshot,,
0,4000.0,-0.0
1,5000.0,2000.0
2,6000.0,1000.0
3,7000.0,-0.0
4,5000.0,2000.0
5,3000.0,-0.0
